In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm
from pathlib import Path

In [2]:
class CrackHybridLoss(nn.Module):
    def __init__(self, class_weights, device):
        super(CrackHybridLoss, self).__init__()
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)
        self.device = device
        
    def forward(self, logits, targets):
        
        ce_loss = self.ce_loss(logits, targets)
        
        probs = F.softmax(logits, dim=1)
        num_classes = logits.shape[1]
        
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        intersection = (probs * targets_one_hot).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))
        
        dice = (2. * intersection + 1e-6) / (union + 1e-6)
        
        dice_loss = 1 - dice[:, 1:].mean() 
        
        return ce_loss + dice_loss

In [3]:
class SegFormerDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512, 512)):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        
        self.mask_filenames = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        self.image_path_map = {}
        for path in Path(image_dir).rglob("*.jpg"):
            self.image_path_map[path.name] = str(path)
            

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_name = self.mask_filenames[idx]
        img_name = mask_name.replace('.png', '.jpg')

        img_path = self.image_path_map.get(img_name)
        
        if img_path is None:
            raise FileNotFoundError(f"'{img_name}' 파일을 찾을 수 없습니다.")

        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") 

        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        img_tensor = transforms.ToTensor()(image)
        img_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img_tensor)

        mask_tensor = torch.tensor(np.array(mask), dtype=torch.long)

        return img_tensor, mask_tensor

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

IMAGE_DIR = r"D:\Study\HumanStudy\Dataset\Training\01.원천데이터"
MASK_DIR = r"D:\Study\HumanStudy\Dataset\Training_Masks"

train_dataset = SegFormerDataset(IMAGE_DIR, MASK_DIR, img_size=(512, 512))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=11,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)
model = model.to(device)

cuda


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
weights = torch.tensor([
    1.4745,  # Class 0
    24.6374,  # Class 1
    12.9677,  # Class 2
    18.5382,  # Class 3
    24.0637,  # Class 4
    24.6504,  # Class 5
    24.7175,  # Class 6
    21.8520,  # Class 7
    25.4713,  # Class 8
    25.1452,  # Class 9
    25.4051,  # Class 10
], dtype=torch.float32).to(device)
criterion = CrackHybridLoss(class_weights=weights, device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0006)

best_loss = float('inf')
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        
        outputs = model(pixel_values=images)
        logits = outputs.logits 
        
        logits_resized = nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        
        loss = criterion(logits_resized, masks.long())
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} 평균 Loss: {epoch_loss:.4f}")
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), 'best_segformer_b0_crack_hybrid_c104.pth')

Epoch 1/10: 100%|█████████████████████████████████████████████████| 12300/12300 [2:33:13<00:00,  1.34it/s, loss=2.1203]


Epoch 1 평균 Loss: 1.7013


Epoch 2/10: 100%|█████████████████████████████████████████████████| 12300/12300 [2:43:43<00:00,  1.25it/s, loss=1.4367]


Epoch 2 평균 Loss: 1.5955


Epoch 3/10: 100%|█████████████████████████████████████████████████| 12300/12300 [2:29:38<00:00,  1.37it/s, loss=1.2542]


...

Epoch 8 평균 Loss: 1.3802


Epoch 9/10: 100%|█████████████████████████████████████████████████| 12300/12300 [2:29:33<00:00,  1.37it/s, loss=1.3628]


Epoch 9 평균 Loss: 1.3687


Epoch 10/10: 100%|████████████████████████████████████████████████| 12300/12300 [2:36:35<00:00,  1.31it/s, loss=1.5556]

Epoch 10 평균 Loss: 1.3549
